# 世界モデルと深層生成モデル

この notebook では、世界モデルと深層生成モデルがどちらも未来や観測を作るのに、何のためにそれを作っているかが違うことを整理します。似た部品を持っていても、目的が違うと設計の重心も変わります。


## 生成と予測は似ているが、仕事が違う

深層生成モデルは『もっともらしい観測を作る』ことに強く、世界モデルは『行動を入れたら次にどう変わるかを追う』ことに強みがあります。両者はどちらも潜在変数を扱いますが、使いどころは同じではありません。


画像をきれいに復元できても、行動を変えたときの未来が追えなければ計画には使えません。逆に状態の進み方だけ当たっても、観測へ戻せなければモデルが何を理解しているか確かめにくいままです。ここでは小さな潜在状態 `z` を介して、生成モデル的な復元と世界モデル的な遷移予測がどこで接続し、どこで分かれるのかを見ます。


## 潜在状態を経由する理由

観測 `obs` をそのまま未来へ送る代わりに、いったん `z` に要約してから遷移させます。こうすると『世界がどう変わるか』と『その状態がどう見えるか』を別々に点検できます。以降のセルでは `A` と `B` で状態の進み方を作り、`decode` で観測へ戻し、最後にロールアウトと計画評価までつなげます。


## 最初の見どころ

行動を変えると `z_next` がどう変わるか、同じ初期状態から複数ステップ回すと誤差がどう積もるか、最後に計画候補の点数づけがどこまで信用できるかを順に追います。


この notebook で使う係数やスコアは最小の教材用設定です。ただし、潜在状態・遷移・観測復元・計画評価という分解自体は、本格的な世界モデルでもそのまま重要です。


## 予測を読むときの姿勢

1 ステップ先が当たっていても、ロールアウトを伸ばすとすぐ崩れることがあります。平均誤差だけで安心せず、どの時点から崩れ始めるか、行動を変えたときに予測の向きが素直に変わるかを見ます。


## ここで押さえたい境界

この notebook は『世界モデルが生成モデルの考え方をどう取り込むか』を見る入口です。高品質画像生成そのものではなく、予測と計画に必要な最小構造を読むつもりで進めてください。


## 観察 1: 世界モデルの遷移初期化

生成モデルとの関係を見る前に、まず『行動を入れると内部状態がどう進むか』の最小形を定義します。


In [ ]:
z_t = 0.20
a_t = 1.0
A, B = 0.90, 0.20
z_next = A * z_t + B * a_t
print('task = world-models-and-generative-models')
print('z_next =', round(z_next, 6))

この遷移を起点に観測復元へ進みます。



## 観察 2: 観測予測を作る

次に、潜在状態から観測を復元する写像を作ります。内部で何が起きているかと、それが外からどう見えるかの役割分担をコードで掴みます。


In [ ]:
def decode(z):
    return {'position': 2.5 * z + 0.1, 'velocity': 0.8 * z - 0.05}
obs_next = decode(z_next)
print('obs_next =', {k: round(v, 4) for k, v in obs_next.items()})
print('keys =', list(obs_next.keys()))

観測予測を別関数に切ると、状態の進み方の誤差と、見え方の誤差を分離して調整できます。



## 計算の対応表

1. $z_{t+1} = f_{\theta}(z_t, a_t)$
2. $\hat{o}_{t+1} = g_{\theta}(z_{t+1})$


## 観察 3: ロールアウトを試す

ここで複数ステップ予測を実行します。1ステップでは見えない誤差の積み上がりを把握するためです。


In [ ]:
actions = [0.0, 1.0, 1.0, 0.0, -0.5]
z = 0.1
traj = []
for a in actions:
    z = 0.92 * z + 0.18 * a
    traj.append(round(z, 5))
print('rollout =', traj)

長期予測で崩れるなら、状態の進み方の安定性や、状態表現の情報量不足を疑います。



## 観察 4: 計画候補を比較する

次に、複数の行動列を比較して、どの計画が望ましいかを評価します。ここで初めて、世界モデルが『先を作るだけでなく、次の行動を決めるために使われる』ことが見えてきます。


In [ ]:
plans = [[0, 1, 1], [1, 1, 1], [0, 0, 1]]
def score_plan(plan):
    z = 0.1
    for a in plan:
        z = 0.92 * z + 0.18 * a
    return z
scores = [round(score_plan(p), 5) for p in plans]
print('scores =', scores)

この `score_plan` は、最終潜在値が高いほど良いと仮定した教育用の代理スコアです。実際の計画では、予測した状態から報酬・コスト・制約違反を計算して評価します。ここで使う `A` と `B` も、本番ではデータから学習される係数です。


計画評価が可能になると、実環境での試行回数を抑えた探索がしやすくなります。



## 観察 5: モデル誤差を監視する

最後に、予測と実測の差を定量化します。世界モデルは『どこまで信用してよいか』を常に点検する運用が重要です。


In [ ]:
pred = [0.10, 0.22, 0.31, 0.29]
real = [0.11, 0.25, 0.28, 0.35]
errors = [abs(p - r) for p, r in zip(pred, real)]
print('errors =', [round(e, 4) for e in errors])
print('mean_error =', round(sum(errors) / len(errors), 5))

平均誤差だけでなく時点別誤差を追うと、どの条件でモデルが弱いかを特定しやすくなります。



## 読み終えたあとに残したい視点

1. もっともらしく作れることと、行動を決めるために使えることは同じではない。
2. 潜在状態は圧縮であると同時に、計画の作業場所でもある。
3. 長期ロールアウトを見るまでは、状態の進み方の良し悪しは判断しにくい。
